# Noun evidence G — global supervision control
**Internet ON · GPU T4×2 · Save & Run All.** Một notebook, một nhánh, cả hai GPU.
Đóng băng toàn bộ checkpoint A; chỉ học head108.752tham số từ196patchCLIP→80lớp.
Cộng bằng chứng ảnh vào logits của từ đối tượng, có chặn hot+dog. Không ép chèn từ.
CE + .05SCST(CIDEr-D+.5object_count_v2) sau200updates +.1presence +.1region.
2.500updates, hai vòng train10k, seed42, AdamW1e-4. G/S chỉ khác pooling trong region loss.
Đánh giá A và endpoint rawgreedy trên val5000 đã dùng trước; **không dùng test**.
Gửi ZIP review, giữ head.bin và checkpoint A riêng. Đây là thực nghiệm, chưa có bằng chứng cải thiện.

In [ ]:
print('NOUN EVIDENCE G: START',flush=True)
from pathlib import Path
ARM='G'
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/pair-source-fullframe-ce/upt_hidden_caption_A/checkpoint.bin')
LEGACY_PROMPT=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
OUTPUT=Path('/kaggle/working/noun_evidence_G')
for path in (CHECKPOINT,LEGACY_PROMPT,COCO_JSON,DETECTIONS):
    assert path.is_file(),f'Missing exact input: {path}. Sửa path nếu tài khoản bạn mount khác.'
assert VISUAL_CACHE.is_dir(),VISUAL_CACHE
assert not OUTPUT.exists(),'Preserve old output; use a fresh session/output.'


In [ ]:
import base64, hashlib, json, os, shutil, subprocess, sys, tempfile, zlib
os.environ.update(PYTHONUNBUFFERED='1', HDF5_USE_FILE_LOCKING='FALSE', PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_XET='1')
# Isolate each bootstrap: embedded tracked overlays from a prior run are intentional edits.
REPO = Path(tempfile.mkdtemp(prefix='Image_Captioning-hidden-noun-evidence-g-'))/'repo'
LOG = Path('/tmp/upt_hidden_noun-evidence-g.log')
PROJECT_COMMIT = 'cf248b1b46f8a3a7abd717354308a2c5aa37e2ee'
def run(args, cwd=None):
    args = list(map(str, args))
    print('\nRunning:', ' '.join(args), flush=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nRunning: ' + ' '.join(args) + '\n'); log.flush()
        with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f'Command failed ({status}). Saved log: {LOG}')
def checkout(url, path, revision):
    if not path.exists():
        run(['git','clone',url,path])
    assert (path/'.git').is_dir(), path
    assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=path,text=True).strip(), 'Preserve tracked edits; use a fresh runtime path.'
    run(['git','fetch','--depth','1','origin',revision], cwd=path)
    run(['git','checkout','--detach',revision], cwd=path)
checkout('https://github.com/Supzxjee/Image_Captioning.git', REPO, PROJECT_COMMIT)
SOURCES = json.loads(zlib.decompress(base64.b64decode('...')).decode('utf-8'))
SOURCE_HASHES = {'evidence_data.py': '7043d11c63d7ebfec8c8ed9e81eb951b9954399ff5939d146c35bc6e7ecfdc4e', 'evidence_model.py': '4135abadfef3a90139285905220beff1436d78c7fcdc943b858831b74f443527', 'evidence_stage.py': 'f7f2a4bf3f1229b544101f079566d11494f428cb7d2a721743dde46fec4e62e4', 'evidence_metrics.py': 'e98d18e469415c80fe1c05388b3a4044ce735b88ead039e5cc44fb2b2f4ff453', 'run_evidence.py': '6dd9623304bfabfd0c56ba302ac67aecf0f835d3ab9aa6573ca4b583e5cb557b', 'A_secondary_reference.json': '3635ababe5099669ba5e7758bb58121a60eb087652557fdd933b75a4067bb266', 'pilot_data.py': '93fa39b1de581d774417840be4d5e0b2b8d52bbf9ee53b479adad2c60a45d3d7', 'pilot_model.py': '879046e943169c30e640fd9b04d6a1edfccd19cd17b5311b104b9ec2dc5bf8f1', 'run_pilot.py': 'c34b7d7f2aae7e050190f713370d0c5f85b283bf342e328e63baeaf2783061a1', 'hidden_metric_smoke.py': '8cf64c56b169cdd2407fdeb668408a80d947578632154405c47c7040558a698b', 'captioning/scst_runtime.py': '3452ccfbd43876dc6381b78d1229d1d6b05fe723e3863a04f31c4a784497f669', 'captioning/models.py': '18026877836052c83e75c0d896adcc89822bb35616173c7e8b14b17c9a22e9d8'}
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == SOURCE_HASHES[name], name
    destination = REPO/name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')
print('Pinned runtime and hash-checked embedded source READY', flush=True)


In [ ]:
# Five small label JSON files are bundled with the pinned source commit (~0.96MB ZIP).
# No extra Kaggle dataset/upload is needed; optional dataset copy must have identical receipt.
import zipfile
LABEL_ARCHIVE=REPO/'experiments/noun_evidence_proposal/data/noun_evidence_train10k_v1_dataset.zip'
assert hashlib.sha256(LABEL_ARCHIVE.read_bytes()).hexdigest()=='ebf880fcc0ae23388057abc84ce796b64ee1f0f6c16f804d1cdbf98e73fb78fd'
LABELS=REPO/'noun-evidence-labels'
LABELS.mkdir()
with zipfile.ZipFile(LABEL_ARCHIVE) as archive:
    assert set(archive.namelist())=={'groups.json','targets.json','lexicon.json','prior.json','receipt.json'}
    archive.extractall(LABELS)
assert hashlib.sha256((LABELS/'receipt.json').read_bytes()).hexdigest()=='e315b1148e0034aaec64ea5bdf69f89a3dbea5ecf17183a3d30dc1bd5dcbaa49'
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','pycocotools','pycocoevalcap==1.2',
    'h5py','numpy','Pillow','tqdm','nltk==3.9.2','future'])
run([sys.executable,'-m','pip','install','-q','--no-deps',
    'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'])
run([sys.executable,'-u','-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"])
run(['java','-version'])
CHAIR=Path('/tmp/noun-evidence-chair')
checkout('https://github.com/Maxlinn/CHAIR-metric-standalone.git',CHAIR,'4087a26211aa2339b9a76307cb8f0321ef691d0a')
import torch
assert torch.cuda.device_count()==2,'Select GPU T4x2 for this one arm.'
run([sys.executable,'-u','hidden_metric_smoke.py',CHAIR],cwd=REPO)


In [ ]:
import numpy as np
groups=json.loads((LABELS/'groups.json').read_text())
ids=sorted(r['image_id'] for r in groups['val'])
rng=np.random.default_rng(20261023);shuffled=np.asarray(ids)[rng.permutation(len(ids))]
# Prelock a derangement independently of model outputs; fixed-point-free cyclic shift in random order.
donors={str(int(shuffled[i])):int(shuffled[(i+1)%len(ids)]) for i in range(len(ids))}
assert all(int(i)!=d for i,d in donors.items())
CONFIG=Path('/tmp/noun_evidence_'+ARM+'.json')
cfg=dict(arm=ARM,checkpoint=str(CHECKPOINT),legacy_prompt=str(LEGACY_PROMPT),visual_cache=str(VISUAL_CACHE),
    dataset_json=str(COCO_JSON),detections=str(DETECTIONS),labels=str(LABELS),label_receipt_sha256='e315b1148e0034aaec64ea5bdf69f89a3dbea5ecf17183a3d30dc1bd5dcbaa49',
    output=str(OUTPUT),stage='/tmp/noun_evidence_'+ARM+'.h5',chair=str(CHAIR),project_commit=PROJECT_COMMIT,
    implementation_sha256=SOURCE_HASHES,global_batch_size=8,eval_batch_size=16,seed=42,updates=2500,
    head_lr=1e-4,weight_decay=.01,max_gradient_norm=1.,warmup_updates=200,scst_weight=.05,
    object_reward_weight=.5,presence_weight=.1,region_weight=.1,logit_weight=.5,correction_clip=2.,
    shuffle_donors=donors,shuffle_seed=20261023,head_parameters=108752,
    gate=dict(S_minus_A_CIDEr_min=.01,S_minus_A_paired95_lower_gt=0.,S_minus_G_paired95_lower_gt=0.,
        CHAIRs_increase_max=0.,CHAIRi_increase_max=0.,Recall_drop_max=.005),
    selection='Endpoint only; no test, rerank, sweep or automatic promotion',
    control_policy='Evaluate S controls provisionally after S/A numeric gate; interpret only after S/G gate too.')
CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8');PROTOCOL=OUTPUT/'protocol.json'
def distributed(mode):
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2',
        '--max_restarts=0','run_evidence.py',mode,'--config',PROTOCOL],cwd=REPO)
try:
    run([sys.executable,'-u','run_evidence.py','prepare','--config',CONFIG],cwd=REPO)
    distributed('preflight')
    distributed('worker')
    run([sys.executable,'-u','run_evidence.py','evaluate','--config',PROTOCOL],cwd=REPO)
    if ARM=='S' and json.loads((OUTPUT/'run_summary.json').read_text())['S_A_gate']:
        distributed('controls')
        run([sys.executable,'-u','run_evidence.py','evaluate','--config',PROTOCOL],cwd=REPO)
finally:
    if OUTPUT.is_dir():
        if LOG.is_file():shutil.copyfile(LOG,OUTPUT/'notebook.log')
        for name,source in SOURCES.items():
            target=OUTPUT/'source'/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_text(source,encoding='utf-8')
        if PROTOCOL.is_file():run([sys.executable,'-u','run_evidence.py','archive','--config',PROTOCOL],cwd=REPO)
        else:
            shutil.copyfile(CONFIG,OUTPUT/'requested_config.json')
            with zipfile.ZipFile(str(OUTPUT)+'_review.zip','w',zipfile.ZIP_DEFLATED) as z:
                for path in sorted(OUTPUT.rglob('*')):
                    if path.is_file():z.write(path,path.relative_to(OUTPUT))
print('SEND REVIEW:',str(OUTPUT)+'_review.zip',flush=True)
print('KEEP HEAD:',OUTPUT/'head.bin',flush=True)


**Đầu vào:** checkpoint A, legacy prompt cache, visual cache, MSCOCO dataset JSON, detection cache.
Không cần UPT/VG/noun-box dataset hay ảnh gốc. Nhãn TRAIN đóng gói cùng source và kiểm SHA.
Preflight FAIL thì dừng trước training; ZIP có thể tồn tại dù thất bại. Chỉ COMPLETE mới là kết quả.
G/S cần so sánh khớp order, runtime, base hash và paired CI; chưa thể kết luận từ mỗi nhánh riêng.
head.bin chỉ chứa head mới; cần giữ A checkpoint để dùng lại mô hình.